In [2]:
import collections
import re
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# Initialize the client
client = OpenAI()

# Define a logic problem that benefits from step-by-step reasoning
problem_prompt = (
    "Q: A cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, "
    "how many apples do they have now? Let's think step by step.\nA:"
)

# 1. Generate multiple reasoning paths simultaneously
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant that solves math problems step-by-step."},
        {"role": "user", "content": problem_prompt}
    ],
    temperature=0.7,  # Higher temperature encourages diverse reasoning paths
    n=5,              # Generates 5 independent paths in a single API call
    max_tokens=150
)

print(response.choices)






[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content="Let's break down the problem step by step.\n\n1. **Starting number of apples**: The cafeteria initially had 23 apples.\n\n2. **Apples used for lunch**: They used 20 apples to make lunch. \n   - So, we subtract the number of apples used from the initial amount:\n   \\[\n   23 - 20 = 3 \\text{ apples remaining}\n   \\]\n\n3. **Apples bought**: They bought 6 more apples.\n   - We add the newly bought apples to the remaining apples:\n   \\[\n   3 + 6 = 9 \\text{ apples total}\n   \\]\n\n4. **Final count**: Therefore, after using some apples and buying more, the", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)), Choice(finish_reason='length', index=1, logprobs=None, message=ChatCompletionMessage(content="Let's break down the problem step by step.\n\n1. **Initial number of apples**: The cafeteria starts with 23 apples.\n\n2. **Apples used for lunch*

In [3]:
# 2. Collect all reasoning paths
paths = [choice.message.content for choice in response.choices]

In [4]:
print(paths)

["Let's break down the problem step by step.\n\n1. **Starting number of apples**: The cafeteria initially had 23 apples.\n\n2. **Apples used for lunch**: They used 20 apples to make lunch. \n   - So, we subtract the number of apples used from the initial amount:\n   \\[\n   23 - 20 = 3 \\text{ apples remaining}\n   \\]\n\n3. **Apples bought**: They bought 6 more apples.\n   - We add the newly bought apples to the remaining apples:\n   \\[\n   3 + 6 = 9 \\text{ apples total}\n   \\]\n\n4. **Final count**: Therefore, after using some apples and buying more, the", "Let's break down the problem step by step.\n\n1. **Initial number of apples**: The cafeteria starts with 23 apples.\n\n2. **Apples used for lunch**: They use 20 apples to make lunch. \n   - To find out how many apples are left after using 20, we subtract:\n   \\[\n   23 - 20 = 3\n   \\]\n   So, after using 20 apples, they have 3 apples left.\n\n3. **Apples bought**: They then buy 6 more apples.\n   - To find the total number of

In [5]:
# 3. Extract the final numeric answer from each path using Regex
answers = []
for i, path in enumerate(paths):
    print(f"--- Path {i+1} ---")
    print(path)
    
    # Simple regex to find the last number/digit mentioned in the text
    digits = re.findall(r'\d+', path)
    if digits:
        final_answer = digits[-1]
        answers.append(final_answer)
    print(f"Extracted Answer: {final_answer}\n")


--- Path 1 ---
Let's break down the problem step by step.

1. **Starting number of apples**: The cafeteria initially had 23 apples.

2. **Apples used for lunch**: They used 20 apples to make lunch. 
   - So, we subtract the number of apples used from the initial amount:
   \[
   23 - 20 = 3 \text{ apples remaining}
   \]

3. **Apples bought**: They bought 6 more apples.
   - We add the newly bought apples to the remaining apples:
   \[
   3 + 6 = 9 \text{ apples total}
   \]

4. **Final count**: Therefore, after using some apples and buying more, the
Extracted Answer: 4

--- Path 2 ---
Let's break down the problem step by step.

1. **Initial number of apples**: The cafeteria starts with 23 apples.

2. **Apples used for lunch**: They use 20 apples to make lunch. 
   - To find out how many apples are left after using 20, we subtract:
   \[
   23 - 20 = 3
   \]
   So, after using 20 apples, they have 3 apples left.

3. **Apples bought**: They then buy 6 more apples.
   - To find the total

In [6]:
# 4. Perform a majority vote to get the self-consistent answer
answer_counts = collections.Counter(answers)
most_common_answer, vote_count = answer_counts.most_common(1)[0]

print("=== FINAL AGGREGATION ===")
print(f"All extracted answers: {answers}")
print(f"Majority Vote Winner: {most_common_answer} (with {vote_count}/{len(answers)} votes)")

=== FINAL AGGREGATION ===
All extracted answers: ['4', '6', '6', '9', '6']
Majority Vote Winner: 6 (with 3/5 votes)
